# Course 2 · Week 3 — Hands-on: Bias, Variance, and Learning Curves

This is the most practical week in the entire specialization. Master this and you'll know what to try next when your model isn't working.

By the end you'll have:

1. Split data into train / cv / test (and known why all three exist)
2. Computed J_train and J_cv across model complexities and watched the bias-variance trade-off appear
3. Drawn learning curves and diagnosed under-/over-fitting from their shape
4. Run a regularization sweep, picked the best lambda by cv error
5. (Stretch) Used the test set exactly once to get an honest generalization estimate

Estimated time: **45–60 minutes.**


## Setup — noisy samples from a curve

Underlying truth: `y = sin(1.5x)` plus Gaussian noise. We sample 90 points and split them 60/20/20 into train / cv / test. The model only sees train; we evaluate on cv to pick hyperparameters; we touch test exactly once at the end for an honest report.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

def true_fn(x):
    return np.sin(1.5 * x)

# Generate noisy samples from a curvy function
n_total = 90
X_all = np.random.uniform(-3, 3, n_total)
y_all = true_fn(X_all) + np.random.normal(0, 0.25, n_total)

# 60/20/20 train/cv/test split
idx = np.random.permutation(n_total)
n_train = int(n_total * 0.6); n_cv = int(n_total * 0.2)
i_train, i_cv, i_test = idx[:n_train], idx[n_train:n_train+n_cv], idx[n_train+n_cv:]
X_train, y_train = X_all[i_train], y_all[i_train]
X_cv,    y_cv    = X_all[i_cv],    y_all[i_cv]
X_test,  y_test  = X_all[i_test],  y_all[i_test]

print(f"train: {len(X_train)}, cv: {len(X_cv)}, test: {len(X_test)}")


Three colors of dots. The dashed line is the true function — but in real life you never see it. You only see the dots and have to figure out the rest.


In [ ]:
xs = np.linspace(-3.2, 3.2, 200)
plt.figure(figsize=(8, 4))
plt.plot(xs, true_fn(xs), color="black", lw=1.5, ls="--", label="true function (unknown to us)")
plt.scatter(X_train, y_train, color="#4338ca", s=40, alpha=0.7, label="train")
plt.scatter(X_cv,    y_cv,    color="#10b981", s=40, alpha=0.7, label="cv")
plt.scatter(X_test,  y_test,  color="#ef4444", s=40, alpha=0.7, label="test")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Three subsets carved from the same noisy data")
plt.show()


## Quick recap

**High bias** = model too simple → it can't capture the true pattern. J_train AND J_cv both high.
**High variance** = model too complex → it captures noise. J_train low, J_cv high (big gap).

The fix:
- High bias → bigger model, more features, less regularization
- High variance → more data, simpler model, more regularization

Learning curves help you tell the difference: bias plateaus high; variance has a big gap that closes as data grows.


## Exercise 1 — helper functions

`mse` is the only thing left for you to implement. The others are already provided (closed-form linear regression with optional L2 regularization).


In [ ]:
def poly_features(x, degree):
    """Build a feature matrix [x, x^2, ..., x^degree] from a 1D x."""
    return np.column_stack([x**(d+1) for d in range(degree)])

def fit(X, y, lam=0.0):
    """Closed-form linear regression with optional L2 regularization on weights (not bias)."""
    X_aug = np.column_stack([np.ones(len(X)), X])
    n = X_aug.shape[1]
    A = X_aug.T @ X_aug + lam * np.eye(n)
    A[0, 0] -= lam  # don't regularize the bias
    return np.linalg.solve(A, X_aug.T @ y)

def predict(X, theta):
    return np.column_stack([np.ones(len(X)), X]) @ theta

def mse(y_true, y_pred):
    """Mean squared error."""
    # TODO: implement (one line)
    return 0.0


# Sanity check: mse([1,2,3], [1,2,3]) = 0
assert mse(np.array([1,2,3]), np.array([1,2,3])) == 0
# mse([1,2], [3,4]) = mean([(1-3)^2, (2-4)^2]) = mean([4, 4]) = 4
assert abs(mse(np.array([1.,2.]), np.array([3.,4.])) - 4.0) < 1e-9
print("✓ mse() works")


## Exercise 2 — degree sweep (the bias-variance plot)

For each polynomial degree, fit on train and evaluate on train AND cv. Plot both errors.

You should see: **as degree grows, J_train drops monotonically, J_cv U-shapes** — first dropping (high bias resolved), then rising (variance kicking in).


In [ ]:
# Vary polynomial degree from 1 to 12. For each, fit on train, evaluate on train AND cv.
# This is the classic bias-variance diagnostic.

degrees = [1, 2, 4, 8, 12]
results = []

for d in degrees:
    Xtr = poly_features(X_train, d)
    Xcv = poly_features(X_cv,    d)
    # TODO: fit theta, then compute J_train and J_cv using mse
    theta = np.zeros(d + 1)
    Jtr = 0.0
    Jcv = 0.0
    results.append((d, Jtr, Jcv))

print(f"{'degree':>6}  {'J_train':>10}  {'J_cv':>10}")
for d, jtr, jcv in results:
    print(f"{d:>6}  {jtr:>10.4f}  {jcv:>10.4f}")

# Plot
ds = [r[0] for r in results]
jtrs = [r[1] for r in results]
jcvs = [r[2] for r in results]
plt.figure(figsize=(7, 4))
plt.plot(ds, jtrs, "o-", color="#4338ca", lw=2, label="$J_{train}$")
plt.plot(ds, jcvs, "o-", color="#10b981", lw=2, label="$J_{cv}$")
plt.xlabel("polynomial degree (model complexity)"); plt.ylabel("error")
plt.legend(); plt.grid(alpha=0.3); plt.title("Bias-variance trade-off")
plt.show()


## Visual: under-, just-right, and overfit

Three plots side by side. Watch how the red curve hugs the data:

- **Degree 1:** straight line through curvy data — terrible fit (high bias).
- **Degree 4:** smooth curve following the underlying sine pattern (just right).
- **Degree 12:** wiggly curve tortured to pass through specific noisy points (overfit / high variance).


In [ ]:
# Show three fits visually: degree 1 (underfit), degree 4 (good), degree 12 (overfit).
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
xs_plot = np.linspace(-3.2, 3.2, 200)
for ax, d, label in zip(axes, [1, 4, 12], ["degree 1 — underfit", "degree 4 — about right", "degree 12 — overfit"]):
    Xtr = poly_features(X_train, d)
    theta = fit(Xtr, y_train)
    y_plot = predict(poly_features(xs_plot, d), theta)
    ax.plot(xs_plot, true_fn(xs_plot), color="black", ls="--", label="true")
    ax.scatter(X_train, y_train, color="#4338ca", s=30, label="train")
    ax.plot(xs_plot, y_plot, color="#ef4444", lw=2, label=f"fit")
    ax.set_title(label); ax.legend(); ax.grid(alpha=0.3)
    ax.set_ylim(-2.5, 2.5)
plt.tight_layout(); plt.show()


## Exercise 3 — learning curves

Hold model complexity fixed at degree=8. Vary training-set size from 3 examples up to all 54. For each size, fit, then evaluate J_train (on the subset used for fitting) and J_cv (on the full cv set).

Watch:
- **Tiny m**: training error is *zero* (model has more parameters than data — perfect memorization). CV error is astronomical.
- **Larger m**: training error rises (data outpaces model capacity). CV error drops.
- **Eventually**: they converge to similar values (no more improvement available).


In [ ]:
# Learning curves: hold the model complexity fixed at degree=8, vary training-set size m.
# Compute J_train and J_cv for each.

degree = 8
sizes = [3, 6, 10, 20, 30, len(X_train)]
results_lc = []
for size in sizes:
    Xtr = poly_features(X_train[:size], degree)
    Xcv = poly_features(X_cv,           degree)
    # TODO: fit theta on the first `size` examples, compute J_train (on subset) and J_cv (on full cv)
    theta = np.zeros(degree + 1)
    Jtr = 0.0
    Jcv = 0.0
    results_lc.append((size, Jtr, Jcv))

print(f"{'m':>4}  {'J_train':>10}  {'J_cv':>10}")
for s, jtr, jcv in results_lc:
    print(f"{s:>4}  {jtr:>10.4f}  {jcv:>10.4f}")

ms = [r[0] for r in results_lc]
plt.figure(figsize=(7, 4))
plt.plot(ms, [r[1] for r in results_lc], "o-", color="#4338ca", lw=2, label="$J_{train}$")
plt.plot(ms, [r[2] for r in results_lc], "o-", color="#10b981", lw=2, label="$J_{cv}$")
plt.xlabel("training set size m"); plt.ylabel("error")
plt.yscale("symlog")  # data spans many orders of magnitude
plt.legend(); plt.grid(alpha=0.3); plt.title(f"Learning curves at degree={degree}")
plt.show()


## Exercise 4 — regularization sweep

Now run the OPPOSITE experiment. Fix model complexity at *too high* (degree 12) and vary the regularization strength lambda from 0 to 10.

Watch:
- **lambda=0**: full overfitting capacity, J_cv higher than necessary
- **lambda=1**: best J_cv (regularization helps even though degree is too high)
- **lambda=10**: too much regularization, both J_train and J_cv go up (model can no longer fit)

The minimum-J_cv lambda is the optimal regularization strength.


In [ ]:
# Regularization sweep: at degree=12 (overfit territory), try lambdas from 0 to 10.
# Pick the lambda that minimizes J_cv. That's your optimal regularization strength.

degree = 12
lams = [0, 0.001, 0.01, 0.1, 1.0, 10.0]
results_reg = []
for lam in lams:
    Xtr = poly_features(X_train, degree)
    Xcv = poly_features(X_cv,    degree)
    # TODO: fit with regularization (use lam= in fit()), then evaluate J_train and J_cv
    theta = np.zeros(degree + 1)
    Jtr = 0.0
    Jcv = 0.0
    results_reg.append((lam, Jtr, Jcv, float(np.linalg.norm(theta[1:]))))

print(f"{'lambda':>8}  {'J_train':>10}  {'J_cv':>10}  {'||theta||':>10}")
for lam, jtr, jcv, n in results_reg:
    print(f"{lam:>8}  {jtr:>10.4f}  {jcv:>10.4f}  {n:>10.3f}")

# Pick winner
best = min(results_reg, key=lambda r: r[2])
print(f"\nBest lambda by J_cv: {best[0]} (J_cv = {best[2]:.4f})")


## ⭐ Stretch — finally use the test set

You've used cv to pick degree (or lambda). Now lock in your choice, retrain on (just) train, and evaluate **once** on test. That number is your honest performance.

If J_test is way worse than J_cv, that's a sign your CV size was too small and you got lucky on it.


In [ ]:
# STRETCH: evaluate the chosen model on the test set. This is the ONE TIME you get to peek.
# The cv set was used to pick degree and lambda; the test set tells you the true generalization.

# TODO: pick a (degree, lambda) combo from your earlier results — say degree=8, lambda=0
# Fit, evaluate on test, report J_test.


## Wrap-up

The bias-variance lens is the most useful diagnostic in ML. When your model isn't working, ask:
1. Is J_train high? → high bias → bigger model
2. Is the train/cv gap big? → high variance → more data or regularization
3. Is J_train low and gap small? → great, ship it

Three-tier split (train/cv/test) is the discipline that makes results trustworthy. Tune on cv; report on test; never confuse them.
